## Random Forest

The flow is:

1. Train Random Forest on transformed data.
2. Train Random Forest with class weights.
3. Train Random Forest on resampled datasets:
   - ADASYN
   - SMOTE
   - SMOTE-ENN
   - SMOTE-TOMEK
4. Compare all experiments using evaluation metrics.
5. Tune Random Forest hyperparameters.
6. Select the best Random Forest model.
7. Save the best model.
8. Evaluate the final saved model on test data.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()

while not (project_root / "src").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not find project root containing 'src'")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: F:\CUFE\Data Science\Diabetes-Prediction


In [2]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import joblib
from src.diabetes_prediction.transformation.transformation import DataTransformation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
)

In [3]:
train_transformed = pd.read_csv('../../data/preprocessed/train_preprocessed.csv')
x_train_transformed = train_transformed.drop("diabetes", axis=1)
y_train_transformed = train_transformed["diabetes"]

val = pd.read_csv('../../data/preprocessed/validation_preprocessed.csv')
x_val = val.drop("diabetes", axis=1)
y_val = val["diabetes"]

print(f'x_train_transformed: {train_transformed.shape}')
train_transformed.head()

x_train_transformed: (67139, 21)


,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,glucose_hba1c_interaction,...,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag,diabetes
0,0.0,0.0,0.0,0.0,0.0,0.612112,-0.637209,0.000000,-0.677966,-0.459103,...,-0.056612,-0.232854,-0.081827,0,0,0,0,0,0,0
1,0.0,0.0,1.0,0.0,0.0,0.737237,0.818605,0.500000,0.000000,0.411609,...,0.658534,1.018393,0.557564,0,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.399399,-0.229457,0.000000,0.254237,0.382586,...,-0.339001,0.014118,-0.070409,1,0,0,0,0,0,0
3,0.0,1.0,0.0,0.0,0.0,0.874875,1.485271,0.500000,0.254237,0.668865,...,1.258590,1.470922,1.050428,0,0,0,0,1,0,0
4,0.0,0.0,0.0,1.0,0.0,0.687187,-0.395349,-1.642857,1.016949,-0.142480,...,0.148131,-1.008759,1.078972,1,0,0,0,0,0,0


In [4]:
train_ADASYN = pd.read_csv('../../data/imbalance_resolve/ADASYN.csv')
x_train_ADASYN = train_ADASYN.drop("diabetes_target", axis=1)
y_train_ADASYN = train_ADASYN["diabetes_target"]
print(f'train_ADASYN: {train_ADASYN.shape}')

train_smote_enn = pd.read_csv('../../data/imbalance_resolve/train_smote_enn.csv')
x_train_smote_enn = train_smote_enn.drop("diabetes_target", axis=1)
y_train_smote_enn = train_smote_enn["diabetes_target"]
print(f'train_smote_enn: {train_smote_enn.shape}')

train_smote_tomek = pd.read_csv('../../data/imbalance_resolve/train_smote_tomek.csv')
x_train_smote_tomek= train_smote_tomek.drop("diabetes_target", axis=1)
y_train_smote_tomek = train_smote_tomek["diabetes_target"]
print(f'train_smote_tomek: {train_smote_tomek.shape}')

train_smote = pd.read_csv('../../data/imbalance_resolve/train_smote.csv')
x_train_smote = train_smote.drop("diabetes_target", axis=1)
y_train_smote = train_smote["diabetes_target"]
print(f'train_smote: {train_smote.shape}')

train_ADASYN: (122666, 16)
train_smote_enn: (112303, 16)
train_smote_tomek: (121658, 16)
train_smote: (112303, 16)


#### Helper functions for model evaluation

In [5]:
def evaluate_model(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]

    y_pred = (y_proba >= threshold).astype(int)

    results = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred),
        "F1-score": f1_score(y, y_pred),
        "ROC-AUC": roc_auc_score(y, y_proba),
        "PR-AUC": average_precision_score(y, y_proba),
        "MCC": matthews_corrcoef(y, y_pred) 
    }    

    return results

def print_results(title, results):
    print(f"========== {title} ==========")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")
    print()


def evaluate_and_store(results_table, name, model, X, y, threshold=0.5):
    results = evaluate_model(model, X, y, threshold=threshold)
    print_results(name, results)

    row = {"Experiment": name}
    row.update(results)
    results_table.append(row)

## Experiment 1 — Random Forest on Transformed Train Data

In [6]:
rf_results_table = []

model_rf_transformed = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    random_state=42,
    n_jobs=-1,
)

model_rf_transformed.fit(x_train_transformed, y_train_transformed)

evaluate_and_store(
    rf_results_table,
    "Random Forest - transformed",
    model_rf_transformed,
    x_val,
    y_val,
)


========== Random Forest - transformed ==========
Accuracy: 0.9697
Precision: 0.9428
Recall: 0.6997
F1-score: 0.8032
ROC-AUC: 0.9672
PR-AUC: 0.8663
MCC: 0.7975



## Experiment 2 — Random Forest on Transformed Train Data with Class Weights

In [7]:
model_rf_transformed_w = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model_rf_transformed_w.fit(x_train_transformed, y_train_transformed)

evaluate_and_store(
    rf_results_table,
    "Random Forest - transformed - class_weight balanced",
    model_rf_transformed_w,
    x_val,
    y_val,
    
)


========== Random Forest - transformed - class_weight balanced ==========
Accuracy: 0.9703
Precision: 0.9607
Recall: 0.6918
F1-score: 0.8044
ROC-AUC: 0.9650
PR-AUC: 0.8619
MCC: 0.8012



## Experiment 3 — Random Forest on ADASYN Train Data

In [8]:
model_rf_adasyn = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    random_state=42,
    n_jobs=-1,
)

model_rf_adasyn.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    rf_results_table,
    "Random Forest - ADASYN",
    model_rf_adasyn,
    x_val[x_train_ADASYN.columns],
    y_val,
    
)

========== Random Forest - ADASYN ==========
Accuracy: 0.9679
Precision: 0.8009
Recall: 0.8475
F1-score: 0.8235
ROC-AUC: 0.9841
PR-AUC: 0.9201
MCC: 0.8063



## Experiment 4 — Random Forest on SMOTE Train Data

In [9]:
model_rf_smote = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    random_state=42,
    n_jobs=-1,
)

model_rf_smote.fit(x_train_smote, y_train_smote)

evaluate_and_store(
    rf_results_table,
    "Random Forest - SMOTE",
    model_rf_smote,
    x_val[x_train_smote.columns],
    y_val,
)

========== Random Forest - SMOTE ==========
Accuracy: 0.9343
Precision: 0.5832
Recall: 0.9009
F1-score: 0.7081
ROC-AUC: 0.9775
PR-AUC: 0.8883
MCC: 0.6932



## Experiment 5 — Random Forest on SMOTE-ENN Train Data

In [10]:
model_rf_smote_enn = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    random_state=42,
    n_jobs=-1,
)

model_rf_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)

evaluate_and_store(
    rf_results_table,
    "Random Forest - SMOTE-ENN",
    model_rf_smote_enn,
    x_val[x_train_smote_enn.columns],
    y_val,
)

========== Random Forest - SMOTE-ENN ==========
Accuracy: 0.9343
Precision: 0.5832
Recall: 0.9009
F1-score: 0.7081
ROC-AUC: 0.9775
PR-AUC: 0.8883
MCC: 0.6932



## Experiment 6 — Random Forest on SMOTE-TOMEK Train Data

In [11]:
model_rf_smote_tomek = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    random_state=42,
    n_jobs=-1,
)

model_rf_smote_tomek.fit(x_train_smote_tomek, y_train_smote_tomek)

evaluate_and_store(
    rf_results_table,
    "Random Forest - SMOTE-TOMEK",
    model_rf_smote_tomek,
    x_val[x_train_smote_tomek.columns],
    y_val,
)

========== Random Forest - SMOTE-TOMEK ==========
Accuracy: 0.9698
Precision: 0.8262
Recall: 0.8333
F1-score: 0.8297
ROC-AUC: 0.9831
PR-AUC: 0.9186
MCC: 0.8132



In [12]:
df = pd.DataFrame(rf_results_table)

df.head(10)

,Experiment,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC,MCC
0,Random Forest - transformed,0.969695,0.942797,0.699686,0.803249,0.967190,0.866338,0.797505
1,Random Forest - transformed - class_weight bal...,0.970251,0.960699,0.691824,0.804388,0.965036,0.861945,0.801215
2,Random Forest - ADASYN,0.967888,0.800892,0.847484,0.823529,0.984108,0.920146,0.806272
3,Random Forest - SMOTE,0.934316,0.583206,0.900943,0.708063,0.977525,0.888289,0.693189
4,Random Forest - SMOTE-ENN,0.934316,0.583206,0.900943,0.708063,0.977525,0.888289,0.693189
5,Random Forest - SMOTE-TOMEK,0.969764,0.826189,0.833333,0.829746,0.983124,0.918600,0.813164


## Random Forest Experiments Evaluation Results

The table below compares the performance of different Random Forest experiments on the validation set.

| Experiment | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC |
|---|---:|---:|---:|---:|---:|---:|---:|
| Random Forest - transformed | 0.9697 | 0.9428 | 0.6997 | 0.8032 | 0.9672 | 0.8663 | 0.7975 |
| Random Forest - transformed - class_weight balanced | `0.9703` | `0.9607` | 0.6918 | 0.8044 | 0.9650 | 0.8619 | 0.8012 |
| Random Forest - ADASYN | 0.9679 | 0.8009 | 0.8475 | 0.8235 | `0.9841` | `0.9201` | 0.8063 |
| Random Forest - SMOTE | 0.9343 | 0.5832 | `0.9009` | 0.7081 | 0.9775 | 0.8883 | 0.6932 |
| Random Forest - SMOTE-ENN | 0.9343 | 0.5832 | `0.9009` | 0.7081 | 0.9775 | 0.8883 | 0.6932 |
| Random Forest - SMOTE-TOMEK | 0.9698 | 0.8262 | 0.8333 | `0.8297` | 0.9831 | 0.9186 | 0.8132 |


#### I will continue with Random Forest - ADASYN

#### Tuning hyperparameters using x_train_adasyn

#### Experiment 1

In [13]:
rf_tuning_results = []

model1_adasyn = RandomForestClassifier(
    n_estimators= 100,
    max_depth= 10,
    min_samples_split= 2,
    min_samples_leaf= 1,
    max_features= "sqrt",
    class_weight= None,
    random_state= 42,
    n_jobs= -1,
)

model1_adasyn.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    rf_tuning_results,
    "Random Forest - ADASYN - E1 Tunning",
    model1_adasyn,
    x_val[x_train_ADASYN.columns],
    y_val,
    
)

========== Random Forest - ADASYN - E1 Tunning ==========
Accuracy: 0.8409
Precision: 0.3521
Recall: 0.9520
F1-score: 0.5141
ROC-AUC: 0.9731
PR-AUC: 0.8663
MCC: 0.5206



#### Experiment 2

In [14]:
model2_adasyn = RandomForestClassifier(
    n_estimators= 200,
    max_depth= 15,
    min_samples_split= 5,
    min_samples_leaf= 2,
    max_features= "sqrt",
    class_weight= None,
    random_state= 42,
    n_jobs= -1,
)

model2_adasyn.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    rf_tuning_results,
    "Random Forest - ADASYN - E2 Tunning",
    model2_adasyn,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== Random Forest - ADASYN - E2 Tunning ==========
Accuracy: 0.8858
Precision: 0.4325
Recall: 0.9340
F1-score: 0.5912
ROC-AUC: 0.9760
PR-AUC: 0.8798
MCC: 0.5887



#### Experiment 3


In [15]:
model3_adasyn = RandomForestClassifier(
    n_estimators= 300,
    max_depth= 20,
    min_samples_split= 10,
    min_samples_leaf= 4,
    max_features= "sqrt",
    class_weight= None,
    random_state= 42,
    n_jobs= -1,
)

model3_adasyn.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    rf_tuning_results,
    "Random Forest - ADASYN - E3 Tunning",
    model3_adasyn,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== Random Forest - ADASYN - E3 Tunning ==========
Accuracy: 0.9183
Precision: 0.5219
Recall: 0.9009
F1-score: 0.6609
ROC-AUC: 0.9790
PR-AUC: 0.8956
MCC: 0.6480



#### Experiment 4

In [16]:
model4_adasyn = RandomForestClassifier(
    n_estimators= 300,
    max_depth= None,
    min_samples_split= 5,
    min_samples_leaf= 2,
    max_features= "log2",
    class_weight= None,
    random_state= 42,
    n_jobs= -1,
)

model4_adasyn.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    rf_tuning_results,
    "Random Forest - ADASYN - E4 Tunning",
    model4_adasyn,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== Random Forest - ADASYN - E4 Tunning ==========
Accuracy: 0.9598
Precision: 0.7330
Recall: 0.8569
F1-score: 0.7901
ROC-AUC: 0.9822
PR-AUC: 0.9129
MCC: 0.7709



#### Experiment 5

In [17]:
model5_adasyn = RandomForestClassifier(
    n_estimators= 500,
    max_depth= 20,
    min_samples_split= 5,
    min_samples_leaf= 2,
    max_features= "sqrt",
    class_weight= None,
    random_state= 42,
    n_jobs= -1,
)

model5_adasyn.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    rf_tuning_results,
    "Random Forest - ADASYN - E5 Tunning",
    model5_adasyn,
    x_val[x_train_ADASYN.columns],
    y_val,
)

========== Random Forest - ADASYN - E5 Tunning ==========
Accuracy: 0.9286
Precision: 0.5609
Recall: 0.8876
F1-score: 0.6874
ROC-AUC: 0.9794
PR-AUC: 0.8984
MCC: 0.6712



#### Experiment 6

In [18]:
model6_adasyn = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight=None,
    random_state=42,
    n_jobs=-1,
)



model6_adasyn.fit(x_train_ADASYN, y_train_ADASYN)

evaluate_and_store(
    rf_tuning_results,
    "Random Forest - ADASYN - E6 Tunning",
    model6_adasyn,
    x_val[x_train_ADASYN.columns],
    y_val,
    
)

========== Random Forest - ADASYN - E6 Tunning ==========
Accuracy: 0.9679
Precision: 0.8009
Recall: 0.8475
F1-score: 0.8235
ROC-AUC: 0.9841
PR-AUC: 0.9201
MCC: 0.8063



In [19]:
print(rf_tuning_results)

[{'Experiment': 'Random Forest - ADASYN - E1 Tunning', 'Accuracy': 0.84089803294641, 'Precision': 0.35213724920034895, 'Recall': 0.9520440251572327, 'F1-score': 0.5141158989598811, 'ROC-AUC': 0.9731108097933855, 'PR-AUC': 0.8663153868621328, 'MCC': 0.5206442187837618}, {'Experiment': 'Random Forest - ADASYN - E2 Tunning', 'Accuracy': 0.8857996802669076, 'Precision': 0.4324717874044412, 'Recall': 0.9339622641509434, 'F1-score': 0.5911918387658621, 'ROC-AUC': 0.975965335673541, 'PR-AUC': 0.8797761343598142, 'MCC': 0.5887458300262061}, {'Experiment': 'Random Forest - ADASYN - E3 Tunning', 'Accuracy': 0.9182595398623757, 'Precision': 0.5218579234972678, 'Recall': 0.9009433962264151, 'F1-score': 0.6608996539792388, 'ROC-AUC': 0.978965225376867, 'PR-AUC': 0.8955545667247943, 'MCC': 0.6479967274064737}, {'Experiment': 'Random Forest - ADASYN - E4 Tunning', 'Accuracy': 0.9597553346771391, 'Precision': 0.7330195023537324, 'Recall': 0.8569182389937107, 'F1-score': 0.79014135556361, 'ROC-AUC': 0.

## Random Forest ADASYN Hyperparameter Tuning Results



| Experiment | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC |
|---|---:|---:|---:|---:|---:|---:|---:|
|Random Forest - ADASYN - E1 Tuning | 0.8409 | 0.3521 | 0.9520 | 0.5141 | 0.9731 | 0.8663 | 0.5206 |
|Random Forest - ADASYN - E2 Tuning | 0.8858 | 0.4325 | 0.9340 | 0.5912 | 0.9760 | 0.8798 | 0.5887 |
|Random Forest - ADASYN - E3 Tuning | 0.9183 | 0.5219 | 0.9009 | 0.6609 | 0.9790 | 0.8956 | 0.6480 |
|Random Forest - ADASYN - E4 Tuning | 0.9598 | 0.7330 | 0.8569 | 0.7901 | 0.9822 | 0.9129 | 0.7709 |
|Random Forest - ADASYN - E5 Tuning | 0.9286 | 0.5609 | 0.8876 | 0.6874 | 0.9794 | 0.8984 | 0.6712 |
|Random Forest - ADASYN - E6 Tuning | 0.9679 | 0.8009 | 0.8475 | 0.8235 | 0.9841 | 0.9201 | 0.8063 |

### Interpretation

The tuning results show a clear trade-off between **Recall** and **Precision**.

Experiments **E1**, **E2**, and **E3** achieved very high recall, meaning they detected a larger number of diabetic patients. However, their precision was relatively low, which means they produced more false positives.

Experiment **E6** achieved the best overall balance. It produced the highest **F1-score**, **ROC-AUC**, **PR-AUC**, and **MCC**, while maintaining strong recall and much better precision than the earlier experiments.

### Selected Model

Based on the validation results, the best Random Forest ADASYN model is:

> **Random Forest - ADASYN - E6 Tuning**

This model provides the strongest balance between detecting diabetic patients and maintaining reliable predictions.

### Best Model Results

| Metric | Value |
|---|---:|
| Accuracy | 0.9679 |
| Precision | 0.8009 |
| Recall | 0.8475 |
| F1-score | 0.8235 |
| ROC-AUC | 0.9841 |
| PR-AUC | 0.9201 |
| MCC | 0.8063 |

In [24]:
transformer = DataTransformation()
transformer.load_preprocessor()
test = pd.read_csv('../../data/split/test.csv')
x_test = test.drop("diabetes", axis=1)
y_test = test["diabetes"]
x_test_transformed = transformer.transform(x_test)
x_test_transformed = x_test_transformed[x_train_ADASYN.columns]

## Final Evaluation On Test Data

In [33]:
rf_final_result = []

evaluate_and_store(
    rf_final_result,
    "Random Forest - ADASYN Final",
    model_rf_adasyn,
    x_test_transformed,
    y_test,
    
)

========== Random Forest - ADASYN Final ==========
Accuracy: 0.9659
Precision: 0.7842
Recall: 0.8483
F1-score: 0.8150
ROC-AUC: 0.9830
PR-AUC: 0.9156
MCC: 0.7970



In [35]:
import joblib

joblib.dump(model_rf_adasyn, "../../models/RF_model.pkl")

['../../models/RF_model.pkl']